# Kacchi Aloo band submission (C)

A wedding's guests go back for seconds when the potatoes per guest fall inside a band. This notebook cleans `aloo_count / guests` (Bangla digits, x10 typos, missing aloo) and predicts 1 inside the band. The cut-points were chosen by the final-selection simulation (E21/E22) from the training data.

In [ ]:
import glob, unicodedata
import numpy as np, pandas as pd

root = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob("../data/train.csv"))[0].rsplit("/", 1)[0]
tr_raw, te_raw = pd.read_csv(f"{root}/train.csv", dtype=str), pd.read_csv(f"{root}/test.csv", dtype=str)

def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan

def aloo_per_guest(d):
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    apg = aloo / guests
    return np.where(apg > 3.0, apg / 10, apg)   # a few aloo counts were typed x10

tr = tr_raw.assign(apg=aloo_per_guest(tr_raw), y=tr_raw.went_back_for_seconds.astype(int)).dropna(subset=["apg"])
te = te_raw.assign(apg=aloo_per_guest(te_raw))
print(f"data from {root}: train {len(tr)} rows with aloo, test {len(te)} rows")

In [ ]:
LO, HI = 1.0, 1.975   # band cut-points, variant C

In [ ]:
def band(x, lo, hi):
    return ((x > lo) & (x < hi)).astype(int)

train_acc = (band(tr.apg.values, LO, HI) == tr.y.values).mean()
pred = band(te.apg.values, LO, HI)
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": pred})
sub.to_csv("submission.csv", index=False)
print(f"band {LO} < aloo/guest < {HI}: train accuracy {train_acc:.4f}; test ones {pred.sum()} / {len(pred)}")
sub.head()